# AMPS I · Week 11 — HW10: P1–P1 vs Taylor–Hood vs MINI
**Due:** before the Week 12 lecture · **Points:** 100 · **Submit:** this notebook (`.ipynb`) with all outputs

**Rules**
- Before submitting: *Runtime → Restart session and run all*. Notebooks that do not run end to end receive 0.
- An error inside a `%%fenicsx` cell does **not** stop *Run all*. Read every output before you submit.
- LLM assistants are encouraged. Fill in the **AI-usage log** at the end.
- Every number you report must come from a cell you ran. For every rate: **prediction first**, then measurement, then explanation.
- Mesh for Q2–Q3: `diagonal=mesh.DiagonalType.left_right` (alternating diagonals — not the lecture's meshes).

| Question | Topic | Points |
|---|---|:---:|
| Q1 | Saddle-point structure, inf-sup, counting, Fortin's bubble | 20 |
| Q2 | One solver, three pairs: Poiseuille, MMS rates, P1–P1, cost | 40 |
| Q3 | Discrete inf-sup constants on the `left_right` mesh | 20 |
| Q4 | LLM debugging: a Taylor–Hood script with four bugs | 20 |

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

## Q1 — Theory (20 pts)

**(a) (6) Saddle point.** $\mathscr K=\begin{pmatrix}A&B^\top\\B&0\end{pmatrix}$ with $A\in\mathbb R^{n\times n}$ SPD and $B\in\mathbb R^{m\times n}$ of full row rank ($m<n$).
Show that $\mathscr K$ is invertible, and that it has exactly $n$ positive and $m$ negative eigenvalues. (Hint: block $LDL^\top$ factorisation with the Schur complement
$S=BA^{-1}B^\top$ and Sylvester's law of inertia.) Why can CG not be used?

**(b) (4)** Show that in finite dimensions the discrete inf-sup condition ($\beta_h>0$) is equivalent to $\ker B^\top=\{0\}$. With $\mathbf u=0$ on all of $\partial\Omega$,
which non-zero $q\in Q_h$ is always in $\ker B^\top$ if $Q_h$ contains the constants, and how is it removed in practice?

**(c) (5) Counting.** On the $N\times N$ `right` mesh ($2N^2$ triangles, $\mathbf u=0$ on $\partial\Omega$): count the interior velocity DOFs of P1, P2 and the pressure DOFs of
P0. For which $N$ does P1–P0 have more pressure DOFs than velocity DOFs? Why is "enough velocity DOFs" still **not sufficient** for stability?

**(d) (5) Fortin and the bubble.** For MINI, let $\Pi_1$ be a bounded interpolant onto $P1^2$ and $b_K$ the cubic bubble on $K$. Show that one can choose
$c_K\in\mathbb R^2$ such that $\Pi_h\mathbf v=\Pi_1\mathbf v+\sum_Kc_Kb_K$ satisfies $b(\mathbf v-\Pi_h\mathbf v,q_h)=0$ for all $q_h\in P1$. (Integrate by parts: what does
$\int_K(\mathbf v-\Pi_h\mathbf v)\cdot\nabla q_h$ depend on?)

> *your answers to (a)–(d)*

## Q2 — One solver, three pairs (40 pts)

Write (or copy from the lab) `stokes(N, pair, diag, u_ex, p_ex)` and `errors(...)`. Manufactured solution (given in the cells):
$\tilde{\mathbf u}=\nabla^\perp\psi$ with $\psi=\sin^2(\pi x)\sin^2(\pi y)$ — divergence-free and zero on $\partial\Omega$ — and $\tilde p=\cos\pi x\cos\pi y$.

**(a) (8)** Check $\nabla\cdot\tilde{\mathbf u}=0$ symbolically (assemble $\int(\nabla\cdot\tilde{\mathbf u})^2$). Poiseuille exactness test for Taylor–Hood on the `left_right` mesh,
$N=8$.

**(b) (16)** Taylor–Hood and MINI, `left_right`, $N=8,16,32,64$: $\|\mathbf u-\mathbf u_h\|_0$, $\|\nabla(\mathbf u-\mathbf u_h)\|_0$, $\|p-p_h\|_0$ (means removed), $\|\nabla\cdot\mathbf u_h\|_0$, the
KSP reasons and the number of DOFs. Predict the rates first; explain every deviation.

**(c) (8)** P1–P1 on the same mesh: reasons, errors, and $\max|p_h|$. Plot $p_h$ for $N=16$ next to Taylor–Hood's (serial cell, then a plotting cell).

**(d) (8) Cost.** From (b): how many DOFs does each of Taylor–Hood and MINI need for $\|\nabla(\mathbf u-\mathbf u_h)\|_0\le10^{-2}$? (Extrapolate with the measured rates.)

In [ ]:
%%fenicsx -np 2
# TODO: Q2(a) divergence check and Poiseuille exactness, -np 2

from mpi4py import MPI
import numpy as np
import ufl
import basix.ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import apply_lifting, assemble_matrix, assemble_vector, set_bc

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def rate(E):
    return np.round(np.log2(np.array(E[:-1]) / np.array(E[1:])), 2).tolist()

def elements(pair):
    """Velocity and pressure elements of a Stokes pair on triangles."""
    c = "triangle"
    if pair == "TH":                                            # Taylor-Hood P2^2 / P1
        return basix.ufl.element("Lagrange", c, 2, shape=(2,)), basix.ufl.element("Lagrange", c, 1)
    if pair == "MINI":                                          # (P1 + cubic bubble)^2 / P1
        E = basix.ufl.enriched_element([basix.ufl.element("Lagrange", c, 1), basix.ufl.element("Bubble", c, 3)])
        return basix.ufl.blocked_element(E, shape=(2,)), basix.ufl.element("Lagrange", c, 1)
    if pair == "P1P1":
        return basix.ufl.element("Lagrange", c, 1, shape=(2,)), basix.ufl.element("Lagrange", c, 1)
    if pair == "P2P0":
        return basix.ufl.element("Lagrange", c, 2, shape=(2,)), basix.ufl.element("DG", c, 0)
    if pair == "P1P0":
        return basix.ufl.element("Lagrange", c, 1, shape=(2,)), basix.ufl.element("DG", c, 0)

# HW10 manufactured solution: u = curl of psi = sin^2(pi x) sin^2(pi y) (divergence-free, zero on the boundary), p with zero mean
U2 = lambda x: ufl.as_vector([2 * ufl.pi * ufl.sin(ufl.pi * x[0]) ** 2 * ufl.sin(ufl.pi * x[1]) * ufl.cos(ufl.pi * x[1]),
                              -2 * ufl.pi * ufl.sin(ufl.pi * x[0]) * ufl.cos(ufl.pi * x[0]) * ufl.sin(ufl.pi * x[1]) ** 2])
P2_ = lambda x: ufl.cos(ufl.pi * x[0]) * ufl.cos(ufl.pi * x[1])

In [ ]:
%%fenicsx -np 2
# TODO: Q2(b) rates for TH and MINI, -np 2

from mpi4py import MPI
import numpy as np
import ufl
import basix.ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import apply_lifting, assemble_matrix, assemble_vector, set_bc

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def rate(E):
    return np.round(np.log2(np.array(E[:-1]) / np.array(E[1:])), 2).tolist()

def elements(pair):
    """Velocity and pressure elements of a Stokes pair on triangles."""
    c = "triangle"
    if pair == "TH":                                            # Taylor-Hood P2^2 / P1
        return basix.ufl.element("Lagrange", c, 2, shape=(2,)), basix.ufl.element("Lagrange", c, 1)
    if pair == "MINI":                                          # (P1 + cubic bubble)^2 / P1
        E = basix.ufl.enriched_element([basix.ufl.element("Lagrange", c, 1), basix.ufl.element("Bubble", c, 3)])
        return basix.ufl.blocked_element(E, shape=(2,)), basix.ufl.element("Lagrange", c, 1)
    if pair == "P1P1":
        return basix.ufl.element("Lagrange", c, 1, shape=(2,)), basix.ufl.element("Lagrange", c, 1)
    if pair == "P2P0":
        return basix.ufl.element("Lagrange", c, 2, shape=(2,)), basix.ufl.element("DG", c, 0)
    if pair == "P1P0":
        return basix.ufl.element("Lagrange", c, 1, shape=(2,)), basix.ufl.element("DG", c, 0)

# HW10 manufactured solution: u = curl of psi = sin^2(pi x) sin^2(pi y) (divergence-free, zero on the boundary), p with zero mean
U2 = lambda x: ufl.as_vector([2 * ufl.pi * ufl.sin(ufl.pi * x[0]) ** 2 * ufl.sin(ufl.pi * x[1]) * ufl.cos(ufl.pi * x[1]),
                              -2 * ufl.pi * ufl.sin(ufl.pi * x[0]) * ufl.cos(ufl.pi * x[0]) * ufl.sin(ufl.pi * x[1]) ** 2])
P2_ = lambda x: ufl.cos(ufl.pi * x[0]) * ufl.cos(ufl.pi * x[1])

> *your prediction, table and explanations for (b)*

In [ ]:
%%fenicsx -np 2
# TODO: Q2(c) P1-P1 rates and the pressure field, -np 2

from mpi4py import MPI
import numpy as np
import ufl
import basix.ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import apply_lifting, assemble_matrix, assemble_vector, set_bc

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def rate(E):
    return np.round(np.log2(np.array(E[:-1]) / np.array(E[1:])), 2).tolist()

def elements(pair):
    """Velocity and pressure elements of a Stokes pair on triangles."""
    c = "triangle"
    if pair == "TH":                                            # Taylor-Hood P2^2 / P1
        return basix.ufl.element("Lagrange", c, 2, shape=(2,)), basix.ufl.element("Lagrange", c, 1)
    if pair == "MINI":                                          # (P1 + cubic bubble)^2 / P1
        E = basix.ufl.enriched_element([basix.ufl.element("Lagrange", c, 1), basix.ufl.element("Bubble", c, 3)])
        return basix.ufl.blocked_element(E, shape=(2,)), basix.ufl.element("Lagrange", c, 1)
    if pair == "P1P1":
        return basix.ufl.element("Lagrange", c, 1, shape=(2,)), basix.ufl.element("Lagrange", c, 1)
    if pair == "P2P0":
        return basix.ufl.element("Lagrange", c, 2, shape=(2,)), basix.ufl.element("DG", c, 0)
    if pair == "P1P0":
        return basix.ufl.element("Lagrange", c, 1, shape=(2,)), basix.ufl.element("DG", c, 0)

# HW10 manufactured solution: u = curl of psi = sin^2(pi x) sin^2(pi y) (divergence-free, zero on the boundary), p with zero mean
U2 = lambda x: ufl.as_vector([2 * ufl.pi * ufl.sin(ufl.pi * x[0]) ** 2 * ufl.sin(ufl.pi * x[1]) * ufl.cos(ufl.pi * x[1]),
                              -2 * ufl.pi * ufl.sin(ufl.pi * x[0]) * ufl.cos(ufl.pi * x[0]) * ufl.sin(ufl.pi * x[1]) ** 2])
P2_ = lambda x: ufl.cos(ufl.pi * x[0]) * ufl.cos(ufl.pi * x[1])

In [ ]:
%%fenicsx -np 1
# TODO: Q2(c) save p_h (P1-P1 and TH, N = 16) for plotting, -np 1

from mpi4py import MPI
import numpy as np
import ufl
import basix.ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import apply_lifting, assemble_matrix, assemble_vector, set_bc

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def rate(E):
    return np.round(np.log2(np.array(E[:-1]) / np.array(E[1:])), 2).tolist()

def elements(pair):
    """Velocity and pressure elements of a Stokes pair on triangles."""
    c = "triangle"
    if pair == "TH":                                            # Taylor-Hood P2^2 / P1
        return basix.ufl.element("Lagrange", c, 2, shape=(2,)), basix.ufl.element("Lagrange", c, 1)
    if pair == "MINI":                                          # (P1 + cubic bubble)^2 / P1
        E = basix.ufl.enriched_element([basix.ufl.element("Lagrange", c, 1), basix.ufl.element("Bubble", c, 3)])
        return basix.ufl.blocked_element(E, shape=(2,)), basix.ufl.element("Lagrange", c, 1)
    if pair == "P1P1":
        return basix.ufl.element("Lagrange", c, 1, shape=(2,)), basix.ufl.element("Lagrange", c, 1)
    if pair == "P2P0":
        return basix.ufl.element("Lagrange", c, 2, shape=(2,)), basix.ufl.element("DG", c, 0)
    if pair == "P1P0":
        return basix.ufl.element("Lagrange", c, 1, shape=(2,)), basix.ufl.element("DG", c, 0)

# HW10 manufactured solution: u = curl of psi = sin^2(pi x) sin^2(pi y) (divergence-free, zero on the boundary), p with zero mean
U2 = lambda x: ufl.as_vector([2 * ufl.pi * ufl.sin(ufl.pi * x[0]) ** 2 * ufl.sin(ufl.pi * x[1]) * ufl.cos(ufl.pi * x[1]),
                              -2 * ufl.pi * ufl.sin(ufl.pi * x[0]) * ufl.cos(ufl.pi * x[0]) * ufl.sin(ufl.pi * x[1]) ** 2])
P2_ = lambda x: ufl.cos(ufl.pi * x[0]) * ufl.cos(ufl.pi * x[1])

In [ ]:
# TODO: Q2(c) plot the two pressure fields

> *your answers to (c) and (d)*

## Q3 — Discrete inf-sup constants (20 pts)

Write `infsup(N, pair, diag)` (serial): assemble the vector-Laplacian matrix $A$ on the velocity space, $B$ from $-\int q\,\nabla\cdot\mathbf u$, the pressure mass
matrix $M_p$; restrict to interior velocity DOFs; return the smallest non-zero eigenvalue of $(BA^{-1}B^\top,M_p)$ (its square root is $\beta_h$) and the number
of zero eigenvalues. Hints: `fem.assemble_matrix` (CSR) refuses the rectangular blocked $B$ in 0.11 (*"Non-square blocksize unsupported in Python"*) —
use `dolfinx.fem.petsc.assemble_matrix` and `getValuesCSR()`; blocked DOF $i$ of a vector space has the components `2*i`, `2*i+1`.

**(a) (12)** MINI, Taylor–Hood and P1–P1 on the `left_right` mesh, $N=4,8,16$. Table of $\beta_h$ and zero modes. **(b) (8)** Which pairs are uniformly stable? Compare P1–P1 on
`left_right` with the lecture's `right` and `crossed` numbers: does the number of spurious modes depend on the mesh? What does Q2(c) look like in this light?

In [ ]:
%%fenicsx -np 1
# TODO: Q3 infsup on the left_right mesh, -np 1

from mpi4py import MPI
import numpy as np
import ufl
import basix.ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import apply_lifting, assemble_matrix, assemble_vector, set_bc

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def rate(E):
    return np.round(np.log2(np.array(E[:-1]) / np.array(E[1:])), 2).tolist()

def elements(pair):
    """Velocity and pressure elements of a Stokes pair on triangles."""
    c = "triangle"
    if pair == "TH":                                            # Taylor-Hood P2^2 / P1
        return basix.ufl.element("Lagrange", c, 2, shape=(2,)), basix.ufl.element("Lagrange", c, 1)
    if pair == "MINI":                                          # (P1 + cubic bubble)^2 / P1
        E = basix.ufl.enriched_element([basix.ufl.element("Lagrange", c, 1), basix.ufl.element("Bubble", c, 3)])
        return basix.ufl.blocked_element(E, shape=(2,)), basix.ufl.element("Lagrange", c, 1)
    if pair == "P1P1":
        return basix.ufl.element("Lagrange", c, 1, shape=(2,)), basix.ufl.element("Lagrange", c, 1)
    if pair == "P2P0":
        return basix.ufl.element("Lagrange", c, 2, shape=(2,)), basix.ufl.element("DG", c, 0)
    if pair == "P1P0":
        return basix.ufl.element("Lagrange", c, 1, shape=(2,)), basix.ufl.element("DG", c, 0)

> *your table and answers to (a)–(b)*

## Q4 — LLM debugging: a Taylor–Hood script with four bugs (20 pts)

An LLM was asked for "a Taylor–Hood Stokes solver on the unit square with an MMS and the $L^2$/$H^1$ velocity and $L^2$ pressure errors for $N=8,16,32$". It produced the code
below.

```python
from mpi4py import MPI
import numpy as np, ufl, basix.ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem

def run(N):
    msh = mesh.create_unit_square(MPI.COMM_WORLD, N, N)
    W = fem.functionspace(msh, basix.ufl.mixed_element([basix.ufl.element("Lagrange", "triangle", 2, shape=(2,)),
                                                          basix.ufl.element("Lagrange", "triangle", 1)]))
    x = ufl.SpatialCoordinate(msh)
    u_ex = ufl.as_vector([ufl.sin(ufl.pi * x[0]) * ufl.sin(ufl.pi * x[1]), ufl.sin(ufl.pi * x[0]) * ufl.sin(ufl.pi * x[1])])
    p_ex = x[0] + x[1]
    f = -ufl.div(ufl.grad(u_ex)) + ufl.grad(p_ex)
    (u, p), (v, q) = ufl.TrialFunctions(W), ufl.TestFunctions(W)
    a = (ufl.inner(ufl.grad(u), ufl.grad(v)) - p * ufl.div(v) - q * ufl.div(u)) * ufl.dx
    L = ufl.inner(f, v) * ufl.dx
    V0, _ = W.sub(0).collapse()
    u_bc = fem.Function(V0); u_bc.interpolate(fem.Expression(u_ex, V0.element.interpolation_points))
    msh.topology.create_connectivity(1, 2)
    dofs = fem.locate_dofs_topological((W.sub(0), V0), 1, mesh.exterior_facet_indices(msh.topology))
    wh = LinearProblem(a, L, bcs=[fem.dirichletbc(u_bc, dofs, W.sub(0))], petsc_options_prefix=f"stokes{N}_",
                       petsc_options={"ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"}).solve()
    uh, ph = wh.sub(0).collapse(), wh.sub(1).collapse()
    uI = fem.Function(V0); uI.interpolate(fem.Expression(u_ex, V0.element.interpolation_points))
    S = lambda e: MPI.COMM_WORLD.allreduce(fem.assemble_scalar(fem.form(e * ufl.dx)), op=MPI.SUM)
    return [np.sqrt(S(ufl.inner(uh - uI, uh - uI))), np.sqrt(S(ufl.inner(ufl.grad(uh - uI), ufl.grad(uh - uI)))), np.sqrt(S((ph - p_ex) ** 2))]

E = np.array([run(N) for N in (8, 16, 32)])
if MPI.COMM_WORLD.rank == 0:
    print(E, np.log2(E[:-1] / E[1:]))
```

**(a) (8)** Run it (next cell) and find the **four** bugs. For each: the line, what is wrong, the symptom, and how you detected it.
**(b) (8)** Corrected version with a divergence-free MMS of your choice; report the errors and rates for $N=8,16,32$.
**(c) (4)** Which bug would you **not** notice from the printed rates alone — and which check exposes it?

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
import numpy as np, ufl, basix.ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem

def run(N):
    msh = mesh.create_unit_square(MPI.COMM_WORLD, N, N)
    W = fem.functionspace(msh, basix.ufl.mixed_element([basix.ufl.element("Lagrange", "triangle", 2, shape=(2,)),
                                                          basix.ufl.element("Lagrange", "triangle", 1)]))
    x = ufl.SpatialCoordinate(msh)
    u_ex = ufl.as_vector([ufl.sin(ufl.pi * x[0]) * ufl.sin(ufl.pi * x[1]), ufl.sin(ufl.pi * x[0]) * ufl.sin(ufl.pi * x[1])])
    p_ex = x[0] + x[1]
    f = -ufl.div(ufl.grad(u_ex)) + ufl.grad(p_ex)
    (u, p), (v, q) = ufl.TrialFunctions(W), ufl.TestFunctions(W)
    a = (ufl.inner(ufl.grad(u), ufl.grad(v)) - p * ufl.div(v) - q * ufl.div(u)) * ufl.dx
    L = ufl.inner(f, v) * ufl.dx
    V0, _ = W.sub(0).collapse()
    u_bc = fem.Function(V0); u_bc.interpolate(fem.Expression(u_ex, V0.element.interpolation_points))
    msh.topology.create_connectivity(1, 2)
    dofs = fem.locate_dofs_topological((W.sub(0), V0), 1, mesh.exterior_facet_indices(msh.topology))
    wh = LinearProblem(a, L, bcs=[fem.dirichletbc(u_bc, dofs, W.sub(0))], petsc_options_prefix=f"stokes{N}_",
                       petsc_options={"ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"}).solve()
    uh, ph = wh.sub(0).collapse(), wh.sub(1).collapse()
    uI = fem.Function(V0); uI.interpolate(fem.Expression(u_ex, V0.element.interpolation_points))
    S = lambda e: MPI.COMM_WORLD.allreduce(fem.assemble_scalar(fem.form(e * ufl.dx)), op=MPI.SUM)
    return [np.sqrt(S(ufl.inner(uh - uI, uh - uI))), np.sqrt(S(ufl.inner(ufl.grad(uh - uI), ufl.grad(uh - uI)))), np.sqrt(S((ph - p_ex) ** 2))]

E = np.array([run(N) for N in (8, 16, 32)])
if MPI.COMM_WORLD.rank == 0:
    print(E, np.log2(E[:-1] / E[1:]))

In [ ]:
%%fenicsx -np 2
# TODO: Q4 diagnostics, -np 2

> *your table for (a)*

In [ ]:
%%fenicsx -np 2
# TODO: Q4(b) corrected version, -np 2

> *your answer to (c)*

## 🤖 AI-usage log

*Required in every assignment and project. Using an LLM is encouraged and never penalized — an empty or vague log is.*

| # | Tool / model | What I asked (key prompt, shortened) | What it got wrong or missed | How I detected it | How I fixed it |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |

**Code provenance** — which cells/functions were mostly generated, which did you write or substantially change?

> …

**Verification** — how do you know the final result is correct (MMS rates, benchmark value, conservation check, parallel = serial)?

> …

**Reflection (2–3 sentences)** — what would you do differently when prompting next time?

> …